# Player Integrity Analysis

This notebook compares one player's performance to the rest of the player base using the live SQLite database. It is built for investigation, not proof: unusual results can point to patterns worth reviewing, but they are not enough on their own to conclude cheating.

The core comparisons included here are:
- first-try correctness rate
- overall question correctness rate over time
- challenging-question correctness rate
- additional diagnostics such as before-hint accuracy, answer speed, and source/category splits when data is available

In [ ]:
from pathlib import Path
import os
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

In [ ]:
PLAYER_LOOKUP = "ADD_HERE"
TARGET_PLAYER_LABEL = "Target player"
START_DATE = "2026-01-01"
ANALYSIS_SCOPE = "since_start_date"
CHALLENGING_VALUE_QUANTILE = 0.75
DIFFICULT_QUESTION_QUANTILE = 0.25
DIFFICULTY_BUCKETS = 5
ROLLING_WINDOW_DAYS = 14
MIN_QUESTIONS_FOR_PERCENTILE = 15
SHOW_TOP_MATCHES = 10

In [ ]:
def find_project_root(start_path: Path | None = None) -> Path:

    start = (start_path or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:

        if (candidate / "db" / "schema.sql").exists() and (candidate / "src").exists():

            return candidate

    raise FileNotFoundError(
        "Could not locate the jbot project root from the current notebook location."
    )


def resolve_db_path(project_root: Path) -> Path:

    env_db_path = os.getenv("JBOT_DB_PATH")

    if env_db_path:

        candidate = Path(env_db_path)

        resolved = (
            candidate
            if candidate.is_absolute()
            else (project_root / candidate).resolve()
        )

        if resolved.exists():

            return resolved

    restore_db = (project_root / "jbot_restore.db").resolve()

    if restore_db.exists():

        return restore_db

    primary_db = (project_root / "jbot.db").resolve()

    if primary_db.exists():

        return primary_db

    raise FileNotFoundError(
        "Could not find a database. Expected JBOT_DB_PATH, jbot_restore.db, or jbot.db in the project root."
    )


def connect_db() -> tuple[sqlite3.Connection, Path, Path]:

    project_root = find_project_root()

    db_path = resolve_db_path(project_root)

    connection = sqlite3.connect(db_path)

    connection.row_factory = sqlite3.Row

    return connection, project_root, db_path


def resolve_player(players_df: pd.DataFrame, lookup: str) -> pd.Series:

    value = lookup.strip()

    if not value or value == "type player name or id here":

        raise ValueError("Set PLAYER_LOOKUP in the parameter cell first.")

    exact_id = players_df[players_df["player_id"].str.casefold() == value.casefold()]

    if len(exact_id) == 1:

        return exact_id.iloc[0]

    exact_name = players_df[
        players_df["player_name"].fillna("").str.casefold() == value.casefold()
    ]

    if len(exact_name) == 1:

        return exact_name.iloc[0]

    partial = players_df[
        players_df["player_name"]
        .fillna("")
        .str.contains(value, case=False, regex=False)
        | players_df["player_id"].str.contains(value, case=False, regex=False)
    ].sort_values(["player_name", "player_id"])

    if partial.empty:

        raise ValueError(f"No player matched {value!r}.")

    if len(partial) > 1:

        preview = partial[["player_id", "player_name"]].head(SHOW_TOP_MATCHES)

        raise ValueError(
            "Lookup matched multiple players. Refine PLAYER_LOOKUP. Candidates:\n"
            + preview.to_string(index=False)
        )

    return partial.iloc[0]


def percentage(value: float | None) -> str:

    if value is None or pd.isna(value):

        return "n/a"

    return f"{value:.1%}"


def percentage_points(value: float | None) -> str:

    if value is None or pd.isna(value):

        return "n/a"

    return f"{value * 100:+.1f} pp"


def rate_by_player(df: pd.DataFrame, metric_col: str) -> pd.DataFrame:

    grouped = df.groupby(["player_id", "player_name"], as_index=False).agg(
        opportunities=("daily_question_id", "count"),
        successes=(metric_col, "sum"),
    )

    grouped["rate"] = grouped["successes"] / grouped["opportunities"]

    return grouped


def build_benchmark_cohorts(
    scoped_frame: pd.DataFrame, excluded_player_id: str
) -> dict[str, dict[str, object]]:

    comparison_pool = scoped_frame[
        scoped_frame["player_id"] != excluded_player_id
    ].copy()

    if comparison_pool.empty:

        return {
            "top_1_pct": {"ids": set(), "size": 0},
            "top_10_pct": {"ids": set(), "size": 0},
        }

    player_rates = rate_by_player(comparison_pool, "solved")

    eligible = player_rates[
        player_rates["opportunities"] >= MIN_QUESTIONS_FOR_PERCENTILE
    ].sort_values(
        ["rate", "opportunities", "player_id"], ascending=[False, False, True]
    )

    if eligible.empty:

        return {
            "top_1_pct": {"ids": set(), "size": 0},
            "top_10_pct": {"ids": set(), "size": 0},
        }

    top_1_size = max(1, int(np.ceil(len(eligible) * 0.01)))

    top_10_size = max(1, int(np.ceil(len(eligible) * 0.10)))

    return {
        "top_1_pct": {
            "ids": set(eligible.head(top_1_size)["player_id"]),
            "size": top_1_size,
        },
        "top_10_pct": {
            "ids": set(eligible.head(top_10_size)["player_id"]),
            "size": top_10_size,
        },
    }


def metric_summary(
    label: str, df: pd.DataFrame, selected_player_id: str, metric_col: str
) -> dict:

    eligible = df[df[metric_col].notna()].copy()

    if eligible.empty:

        return {
            "metric": label,
            "player_rate": np.nan,
            "field_rate": np.nan,
            "median_player_rate": np.nan,
            "delta_vs_field_pp": np.nan,
            "player_percentile": np.nan,
            "player_n": 0,
        }

    player_rows = eligible[eligible["player_id"] == selected_player_id]

    field_rows = eligible[eligible["player_id"] != selected_player_id]

    player_rate = player_rows[metric_col].mean() if not player_rows.empty else np.nan

    field_rate = field_rows[metric_col].mean() if not field_rows.empty else np.nan

    player_rates = rate_by_player(eligible, metric_col)

    stable_rates = player_rates[
        player_rates["opportunities"] >= MIN_QUESTIONS_FOR_PERCENTILE
    ].copy()

    selected_rate_row = player_rates[player_rates["player_id"] == selected_player_id]

    percentile = np.nan

    if not stable_rates.empty and not selected_rate_row.empty:

        selected_rate = selected_rate_row.iloc[0]["rate"]

        percentile = (stable_rates["rate"] <= selected_rate).mean()

    return {
        "metric": label,
        "player_rate": player_rate,
        "field_rate": field_rate,
        "median_player_rate": (
            player_rates["rate"].median() if not player_rates.empty else np.nan
        ),
        "delta_vs_field_pp": (
            (player_rate - field_rate) * 100
            if pd.notna(player_rate) and pd.notna(field_rate)
            else np.nan
        ),
        "player_percentile": percentile,
        "player_n": len(player_rows),
    }


def metric_counts(
    frame: pd.DataFrame, metric_col: str, player_ids: set[str] | None = None
) -> tuple[int, int]:

    eligible = frame[frame[metric_col].notna()].copy()

    if player_ids is not None:

        eligible = eligible[eligible["player_id"].isin(player_ids)]

    if eligible.empty:

        return 0, 0

    successes = int(eligible[metric_col].sum())

    total = int(len(eligible))

    return successes, total


def wilson_interval(
    successes: int, total: int, z: float = 1.959963984540054
) -> tuple[float, float]:

    if total <= 0:

        return np.nan, np.nan

    phat = successes / total

    denominator = 1 + (z**2 / total)

    center = (phat + (z**2 / (2 * total))) / denominator

    margin = (
        z * np.sqrt((phat * (1 - phat) / total) + (z**2 / (4 * total**2))) / denominator
    )

    return max(0.0, center - margin), min(1.0, center + margin)


def difference_interval(
    successes_a: int,
    total_a: int,
    successes_b: int,
    total_b: int,
) -> tuple[float, float]:

    if total_a <= 0 or total_b <= 0:

        return np.nan, np.nan

    lower_a, upper_a = wilson_interval(successes_a, total_a)

    lower_b, upper_b = wilson_interval(successes_b, total_b)

    return lower_a - upper_b, upper_a - lower_b


def format_interval(interval: tuple[float, float]) -> str:

    low, high = interval

    if pd.isna(low) or pd.isna(high):

        return "n/a"

    return f"[{low * 100:+.1f}, {high * 100:+.1f}] pp"

In [ ]:
PLAYER_QUESTION_SQL = """

WITH morning_messages AS (

    SELECT dq.id AS daily_question_id, MIN(m.timestamp) AS morning_sent_at

    FROM daily_questions dq

    LEFT JOIN messages m

        ON m.status = 'morning_message'

       AND m.timestamp >= dq.sent_at

       AND m.timestamp < date(dq.sent_at, '+2 days')

    GROUP BY dq.id

),

hint_messages AS (

    SELECT
        dq.id AS daily_question_id,
        (
            SELECT m.timestamp
            FROM messages m
            WHERE m.status = 'reminder_message'
              AND m.timestamp > mm.morning_sent_at
              AND m.timestamp < date(dq.sent_at, '+2 days')
            ORDER BY m.timestamp ASC
            LIMIT 1
        ) AS hint_sent_at

    FROM daily_questions dq
    LEFT JOIN morning_messages mm ON mm.daily_question_id = dq.id

    GROUP BY dq.id

),

ranked_guesses AS (

    SELECT

        g.*,

        ROW_NUMBER() OVER (

            PARTITION BY g.daily_question_id, g.player_id

            ORDER BY g.guessed_at ASC, g.id ASC

        ) AS rn

    FROM guesses g

),

player_pool AS (

    SELECT DISTINCT g.player_id

    FROM guesses g

),

player_question AS (

    SELECT

        dq.id AS daily_question_id,

        dq.sent_at,

        q.id AS question_id,

        q.category,

        q.value,

        q.source,

        p.id AS player_id,

        COALESCE(NULLIF(p.name, ''), p.id) AS player_name,

        COUNT(rg.id) AS attempts,

        COALESCE(MAX(CASE WHEN rg.is_correct = 1 THEN 1 ELSE 0 END), 0) AS solved,

        COALESCE(MAX(CASE WHEN rg.rn = 1 AND rg.is_correct = 1 THEN 1 ELSE 0 END), 0) AS first_try_correct,

        MIN(rg.guessed_at) AS first_guess_at,

        MIN(CASE WHEN rg.is_correct = 1 THEN rg.guessed_at END) AS first_correct_at,

        MIN(CASE WHEN rg.rn = 1 THEN rg.guess_text END) AS first_guess_text

    FROM daily_questions dq

    JOIN questions q ON q.id = dq.question_id

    JOIN player_pool pp ON 1 = 1

    JOIN players p ON p.id = pp.player_id

    LEFT JOIN ranked_guesses rg
        ON rg.daily_question_id = dq.id
       AND rg.player_id = p.id

    WHERE p.created_at IS NULL
       OR datetime(p.created_at) <= datetime(date(dq.sent_at, '+1 day'))

    GROUP BY dq.id, dq.sent_at, q.id, q.category, q.value, q.source, p.id, p.name

)

SELECT

    pq.*,

    mm.morning_sent_at,

    hm.hint_sent_at,

    CASE

        WHEN mm.morning_sent_at IS NOT NULL AND pq.first_guess_at IS NOT NULL

        THEN (julianday(pq.first_guess_at) - julianday(mm.morning_sent_at)) * 24 * 60

    END AS minutes_to_first_guess,

    CASE

        WHEN mm.morning_sent_at IS NOT NULL AND pq.first_correct_at IS NOT NULL

        THEN (julianday(pq.first_correct_at) - julianday(mm.morning_sent_at)) * 24 * 60

    END AS minutes_to_correct,

    CASE

        WHEN pq.first_correct_at IS NOT NULL

         AND (hm.hint_sent_at IS NULL OR pq.first_correct_at < hm.hint_sent_at)

        THEN 1 ELSE 0

    END AS correct_before_hint

FROM player_question pq

LEFT JOIN morning_messages mm ON mm.daily_question_id = pq.daily_question_id

LEFT JOIN hint_messages hm ON hm.daily_question_id = pq.daily_question_id

ORDER BY pq.sent_at ASC, pq.player_name ASC

"""
PLAYERS_SQL = """

SELECT

    id AS player_id,

    COALESCE(NULLIF(name, ''), id) AS player_name,

    lifetime_questions,

    lifetime_correct,

    lifetime_first_answers,

    answer_streak,

    season_score,

    score

FROM players

ORDER BY player_name ASC, player_id ASC

"""
SEASONS_SQL = """

SELECT

    season_id,

    season_name,

    start_date,

    end_date,

    is_active

FROM seasons

ORDER BY start_date ASC

"""
conn, project_root, db_path = connect_db()
players = pd.read_sql_query(PLAYERS_SQL, conn)
player_question = pd.read_sql_query(PLAYER_QUESTION_SQL, conn)
seasons = pd.read_sql_query(SEASONS_SQL, conn)
conn.close()


def parse_mixed_timestamp(series: pd.Series) -> pd.Series:

    return pd.to_datetime(series, format="mixed", errors="coerce")


player_question["sent_at"] = parse_mixed_timestamp(player_question["sent_at"])
player_question["first_guess_at"] = parse_mixed_timestamp(
    player_question["first_guess_at"]
)
player_question["first_correct_at"] = parse_mixed_timestamp(
    player_question["first_correct_at"]
)
player_question["morning_sent_at"] = parse_mixed_timestamp(
    player_question["morning_sent_at"]
)
player_question["hint_sent_at"] = parse_mixed_timestamp(player_question["hint_sent_at"])

# Recompute from parsed datetimes so mixed timestamp formats do not skew comparisons.
player_question["correct_before_hint"] = (
    player_question["first_correct_at"].notna()
    & (
        player_question["hint_sent_at"].isna()
        | (player_question["first_correct_at"] < player_question["hint_sent_at"])
    )
).astype(int)
player_question["minutes_to_first_guess"] = pd.to_numeric(
    player_question["minutes_to_first_guess"], errors="coerce"
)
player_question["minutes_to_correct"] = pd.to_numeric(
    player_question["minutes_to_correct"], errors="coerce"
)
seasons["start_date"] = parse_mixed_timestamp(seasons["start_date"])
seasons["end_date"] = parse_mixed_timestamp(seasons["end_date"])
analysis_start_date = pd.Timestamp(START_DATE)
unique_question_values = (
    player_question[["daily_question_id", "value"]].drop_duplicates()["value"].dropna()
)
challenging_threshold = (
    unique_question_values.quantile(CHALLENGING_VALUE_QUANTILE)
    if not unique_question_values.empty
    else np.nan
)
player_question["is_challenging"] = False
if pd.notna(challenging_threshold):

    player_question["is_challenging"] = (
        player_question["value"].fillna(float("-inf")) >= challenging_threshold
    )
question_difficulty = player_question.groupby(
    ["daily_question_id", "sent_at"], as_index=False
).agg(
    field_correct_rate=("solved", "mean"),
    players_answered=("player_id", "nunique"),
    value=("value", "first"),
    category=("category", "first"),
    source=("source", "first"),
)
question_difficulty["difficulty_score"] = 1 - question_difficulty["field_correct_rate"]
question_difficulty["difficulty_percentile"] = question_difficulty[
    "difficulty_score"
].rank(pct=True, method="average")
difficult_question_threshold = question_difficulty["field_correct_rate"].quantile(
    DIFFICULT_QUESTION_QUANTILE
)
question_difficulty["is_difficult_for_field"] = (
    question_difficulty["field_correct_rate"] <= difficult_question_threshold
)
bucket_labels = [f"Q{i}" for i in range(1, DIFFICULTY_BUCKETS + 1)]
effective_buckets = min(
    DIFFICULTY_BUCKETS, question_difficulty["daily_question_id"].nunique()
)
if effective_buckets >= 2:

    bucket_codes = pd.qcut(
        question_difficulty["field_correct_rate"],
        q=effective_buckets,
        labels=False,
        duplicates="drop",
    )

    bucket_count = int(bucket_codes.max()) + 1

    difficulty_labels = bucket_labels[:bucket_count]

    question_difficulty["difficulty_bucket"] = pd.Categorical.from_codes(
        bucket_codes.astype(int),
        categories=difficulty_labels,
        ordered=True,
    )
else:

    question_difficulty["difficulty_bucket"] = "Q1"
player_question = player_question.merge(
    question_difficulty[
        [
            "daily_question_id",
            "field_correct_rate",
            "difficulty_score",
            "difficulty_percentile",
            "is_difficult_for_field",
            "difficulty_bucket",
        ]
    ],
    on="daily_question_id",
    how="left",
)
selected_player = resolve_player(players, PLAYER_LOOKUP)
selected_player_id = selected_player["player_id"]
latest_question_date = player_question["sent_at"].max()
sorted_seasons = seasons.sort_values("start_date").copy()
current_season = pd.DataFrame(columns=seasons.columns)
if not seasons.empty:

    active_seasons = seasons[seasons["is_active"] == 1].copy()

    if not active_seasons.empty:

        current_season = active_seasons.sort_values("start_date").tail(1)

    elif pd.notna(latest_question_date):

        covering = seasons[
            (seasons["start_date"] <= latest_question_date)
            & (seasons["end_date"] >= latest_question_date)
        ].copy()

        if not covering.empty:

            current_season = covering.sort_values("start_date").tail(1)

    if current_season.empty:

        current_season = seasons.sort_values("end_date").tail(1)
current_season_label = "Unavailable"
current_season_start = pd.NaT
current_season_end = pd.NaT
last_season_label = "Unavailable"
last_season_start = pd.NaT
last_season_end = pd.NaT
if not current_season.empty:

    current_season_row = current_season.iloc[0]

    current_season_label = current_season_row["season_name"]

    current_season_start = current_season_row["start_date"]

    current_season_end = current_season_row["end_date"]

    prior_seasons = sorted_seasons[
        sorted_seasons["start_date"] < current_season_start
    ].copy()

    if not prior_seasons.empty:

        last_season_row = prior_seasons.sort_values("start_date").tail(1).iloc[0]

        last_season_label = last_season_row["season_name"]

        last_season_start = last_season_row["start_date"]

        last_season_end = last_season_row["end_date"]
scope_frames = {
    "since_start_date": player_question[
        player_question["sent_at"] >= analysis_start_date
    ].copy(),
}
if pd.notna(current_season_start) and pd.notna(current_season_end):

    scope_frames["current_season"] = player_question[
        player_question["sent_at"].between(current_season_start, current_season_end)
    ].copy()
else:

    scope_frames["current_season"] = player_question.iloc[0:0].copy()
if pd.notna(last_season_start) and pd.notna(last_season_end):

    scope_frames["last_season"] = player_question[
        player_question["sent_at"].between(last_season_start, last_season_end)
    ].copy()
else:

    scope_frames["last_season"] = player_question.iloc[0:0].copy()
if ANALYSIS_SCOPE not in scope_frames:

    raise ValueError(
        f"Invalid ANALYSIS_SCOPE {ANALYSIS_SCOPE!r}. Expected one of {list(scope_frames)}."
    )
analysis_frame = scope_frames[ANALYSIS_SCOPE].copy()
selected_rows = analysis_frame[analysis_frame["player_id"] == selected_player_id].copy()
peer_rows = analysis_frame[analysis_frame["player_id"] != selected_player_id].copy()
if selected_rows.empty:

    raise ValueError(
        f"{TARGET_PLAYER_LABEL} has no guesses recorded for ANALYSIS_SCOPE={ANALYSIS_SCOPE!r}."
    )
display(Markdown(f"**Database:** `{db_path}`"))
display(Markdown(f"**Subject:** `{TARGET_PLAYER_LABEL}` (`{selected_player_id}`)"))
display(Markdown(f"**Analysis scope:** `{ANALYSIS_SCOPE}`"))
display(
    Markdown(
        f"**Since-start-date window:** `{analysis_start_date.date()}` through `{player_question['sent_at'].max().date()}`"
    )
)
if pd.notna(current_season_start) and pd.notna(current_season_end):

    display(
        Markdown(
            f"**Current season:** `{current_season_label}` from `{current_season_start.date()}` through `{current_season_end.date()}`"
        )
    )
else:

    display(
        Markdown("**Current season:** unavailable because no season rows were found.")
    )
if pd.notna(last_season_start) and pd.notna(last_season_end):

    display(
        Markdown(
            f"**Last season:** `{last_season_label}` from `{last_season_start.date()}` through `{last_season_end.date()}`"
        )
    )
else:

    display(
        Markdown(
            "**Last season:** unavailable because no earlier season row was found."
        )
    )
display(
    Markdown(
        f"**Target-player opportunities in active scope (including unanswered):** `{len(selected_rows)}`"
    )
)
display(
    Markdown(
        "**Benchmark cohorts:** within each scope, top 1% and top 10% are defined by overall correct rate among non-target players who answered at least `"
        f"{MIN_QUESTIONS_FOR_PERCENTILE}` questions in that scope."
    )
)
if pd.notna(challenging_threshold):

    display(
        Markdown(
            f"**Challenging question threshold:** top `{CHALLENGING_VALUE_QUANTILE:.0%}` of clue values, starting at value `{challenging_threshold:.0f}`"
        )
    )
else:

    display(
        Markdown(
            "**Challenging question threshold:** unavailable because question values are missing."
        )
    )
if pd.notna(difficult_question_threshold):

    display(
        Markdown(
            f"**Field-difficult question threshold:** questions solved by at most `{difficult_question_threshold:.1%}` of players. Q1 is the hardest bucket and has the lowest median field solve rate."
        )
    )

In [ ]:
def benchmark_rate(frame: pd.DataFrame, metric_col: str, player_ids: set[str]) -> float:

    if not player_ids:

        return np.nan

    benchmark_rows = frame[
        frame["player_id"].isin(player_ids) & frame[metric_col].notna()
    ]

    return benchmark_rows[metric_col].mean() if not benchmark_rows.empty else np.nan


def build_scope_metric_rows(scope_name: str, scoped_frame: pd.DataFrame) -> list[dict]:

    metric_labels = [
        "Overall correct %",
        "First-try correct %",
        "Challenging-question correct %",
        "Difficult-for-field correct %",
        "Challenging-question first-try %",
        "Before-hint correct %",
    ]

    if scoped_frame.empty:

        return [
            {
                "scope": scope_name,
                "metric": label,
                "player_rate": np.nan,
                "field_rate": np.nan,
                "top_1_pct_rate": np.nan,
                "top_10_pct_rate": np.nan,
                "median_player_rate": np.nan,
                "delta_vs_field_pp": np.nan,
                "delta_vs_top_1_pct_pp": np.nan,
                "delta_vs_top_10_pct_pp": np.nan,
                "delta_vs_field_ci_95": "n/a",
                "delta_vs_top_1_pct_ci_95": "n/a",
                "delta_vs_top_10_pct_ci_95": "n/a",
                "player_percentile": np.nan,
                "player_n": 0,
            }
            for label in metric_labels
        ]

    benchmark_sets = build_benchmark_cohorts(scoped_frame, selected_player_id)

    metric_frames = [
        ("Overall correct %", scoped_frame, "solved"),
        ("First-try correct %", scoped_frame, "first_try_correct"),
        (
            "Challenging-question correct %",
            scoped_frame[scoped_frame["is_challenging"]],
            "solved",
        ),
        (
            "Difficult-for-field correct %",
            scoped_frame[scoped_frame["is_difficult_for_field"]],
            "solved",
        ),
        (
            "Challenging-question first-try %",
            scoped_frame[scoped_frame["is_challenging"]],
            "first_try_correct",
        ),
        (
            "Before-hint correct %",
            scoped_frame,
            "correct_before_hint",
        ),
    ]

    scoped_metrics = []

    for label, frame, metric_col in metric_frames:

        row = metric_summary(label, frame, selected_player_id, metric_col)

        top_1_pct_rate = benchmark_rate(
            frame, metric_col, benchmark_sets["top_1_pct"]["ids"]
        )

        top_10_pct_rate = benchmark_rate(
            frame, metric_col, benchmark_sets["top_10_pct"]["ids"]
        )

        row["top_1_pct_rate"] = top_1_pct_rate

        row["top_10_pct_rate"] = top_10_pct_rate

        row["delta_vs_top_1_pct_pp"] = (
            (row["player_rate"] - top_1_pct_rate) * 100
            if pd.notna(row["player_rate"]) and pd.notna(top_1_pct_rate)
            else np.nan
        )

        row["delta_vs_top_10_pct_pp"] = (
            (row["player_rate"] - top_10_pct_rate) * 100
            if pd.notna(row["player_rate"]) and pd.notna(top_10_pct_rate)
            else np.nan
        )

        player_successes, player_total = metric_counts(
            frame, metric_col, {selected_player_id}
        )

        field_successes, field_total = metric_counts(frame, metric_col)

        field_successes -= player_successes

        field_total -= player_total

        top_1_successes, top_1_total = metric_counts(
            frame, metric_col, benchmark_sets["top_1_pct"]["ids"]
        )

        top_10_successes, top_10_total = metric_counts(
            frame, metric_col, benchmark_sets["top_10_pct"]["ids"]
        )

        row["delta_vs_field_ci_95"] = format_interval(
            difference_interval(
                player_successes, player_total, field_successes, field_total
            )
        )

        row["delta_vs_top_1_pct_ci_95"] = format_interval(
            difference_interval(
                player_successes, player_total, top_1_successes, top_1_total
            )
        )

        row["delta_vs_top_10_pct_ci_95"] = format_interval(
            difference_interval(
                player_successes, player_total, top_10_successes, top_10_total
            )
        )

        row["player_successes"] = player_successes

        row["player_total"] = player_total

        row["field_successes"] = field_successes

        row["field_total"] = field_total

        row["top_1_pct_successes"] = top_1_successes

        row["top_1_pct_total"] = top_1_total

        row["top_10_pct_successes"] = top_10_successes

        row["top_10_pct_total"] = top_10_total

        row["top_1_pct_players"] = benchmark_sets["top_1_pct"]["size"]

        row["top_10_pct_players"] = benchmark_sets["top_10_pct"]["size"]

        row["scope"] = scope_name

        scoped_metrics.append(row)

    return scoped_metrics


summary_df = pd.DataFrame(
    build_scope_metric_rows("current_season", scope_frames["current_season"])
    + build_scope_metric_rows("last_season", scope_frames["last_season"])
    + build_scope_metric_rows("since_start_date", scope_frames["since_start_date"])
)


summary_df["player_rate_display"] = summary_df["player_rate"].map(percentage)


summary_df["field_rate_display"] = summary_df["field_rate"].map(percentage)


summary_df["top_1_pct_rate_display"] = summary_df["top_1_pct_rate"].map(percentage)


summary_df["top_10_pct_rate_display"] = summary_df["top_10_pct_rate"].map(percentage)


summary_df["median_player_rate_display"] = summary_df["median_player_rate"].map(
    percentage
)


summary_df["player_percentile_display"] = summary_df["player_percentile"].map(
    percentage
)


summary_df["delta_vs_field_pp"] = summary_df["delta_vs_field_pp"].round(1)


summary_df["delta_vs_top_1_pct_pp"] = summary_df["delta_vs_top_1_pct_pp"].round(1)


summary_df["delta_vs_top_10_pct_pp"] = summary_df["delta_vs_top_10_pct_pp"].round(1)


display(
    summary_df[
        [
            "scope",
            "metric",
            "player_rate_display",
            "field_rate_display",
            "top_1_pct_rate_display",
            "top_10_pct_rate_display",
            "delta_vs_field_pp",
            "delta_vs_top_1_pct_pp",
            "delta_vs_top_10_pct_pp",
            "player_percentile_display",
            "player_n",
        ]
    ].rename(
        columns={
            "player_rate_display": "target_player",
            "field_rate_display": "rest_of_field",
            "top_1_pct_rate_display": "top_1_pct_players",
            "top_10_pct_rate_display": "top_10_pct_players",
            "delta_vs_field_pp": "delta_vs_field_pp",
            "delta_vs_top_1_pct_pp": "delta_vs_top_1_pct_pp",
            "delta_vs_top_10_pct_pp": "delta_vs_top_10_pct_pp",
            "player_percentile_display": "percentile_among_players",
            "player_n": "opportunities",
        }
    )
)


confidence_df = summary_df[
    [
        "scope",
        "metric",
        "delta_vs_field_pp",
        "delta_vs_field_ci_95",
        "delta_vs_top_1_pct_pp",
        "delta_vs_top_1_pct_ci_95",
        "delta_vs_top_10_pct_pp",
        "delta_vs_top_10_pct_ci_95",
    ]
].copy()

display(confidence_df)


benchmark_export_df = summary_df[
    [
        "scope",
        "metric",
        "player_rate",
        "field_rate",
        "top_1_pct_rate",
        "top_10_pct_rate",
        "median_player_rate",
        "delta_vs_field_pp",
        "delta_vs_field_ci_95",
        "delta_vs_top_1_pct_pp",
        "delta_vs_top_1_pct_ci_95",
        "delta_vs_top_10_pct_pp",
        "delta_vs_top_10_pct_ci_95",
        "player_percentile",
        "player_n",
        "player_successes",
        "player_total",
        "field_successes",
        "field_total",
        "top_1_pct_successes",
        "top_1_pct_total",
        "top_10_pct_successes",
        "top_10_pct_total",
        "top_1_pct_players",
        "top_10_pct_players",
    ]
].copy()


benchmark_export_path = (
    project_root / "scripts" / "cheating_player_benchmark_metrics.csv"
)


benchmark_export_df.to_csv(benchmark_export_path, index=False)


display(Markdown(f"**Benchmark CSV export:** `{benchmark_export_path}`"))


cohort_summary_rows = []

cohort_member_rows = []

for scope_name, frame in [
    ("current_season", scope_frames["current_season"]),
    ("last_season", scope_frames["last_season"]),
    ("since_start_date", scope_frames["since_start_date"]),
]:

    benchmark_sets = build_benchmark_cohorts(frame, selected_player_id)

    for cohort_key in ["top_1_pct", "top_10_pct"]:

        player_ids = sorted(benchmark_sets[cohort_key]["ids"])

        cohort_summary_rows.append(
            {
                "scope": scope_name,
                "cohort": cohort_key,
                "player_count": benchmark_sets[cohort_key]["size"],
                "player_ids": ", ".join(player_ids),
            }
        )

        for player_id in player_ids:

            cohort_member_rows.append(
                {
                    "scope": scope_name,
                    "cohort": cohort_key,
                    "player_id": player_id,
                }
            )


cohort_summary_df = pd.DataFrame(cohort_summary_rows)

display(cohort_summary_df)


cohort_member_export_df = pd.DataFrame(cohort_member_rows)

cohort_member_export_path = (
    project_root / "scripts" / "cheating_player_benchmark_cohort_ids.csv"
)

cohort_member_export_df.to_csv(cohort_member_export_path, index=False)

display(Markdown(f"**Benchmark cohort ID CSV export:** `{cohort_member_export_path}`"))


selected_profile = players.loc[players["player_id"] == selected_player_id].copy()


selected_profile["subject"] = TARGET_PLAYER_LABEL


selected_profile["lifetime_correct_pct"] = selected_profile[
    "lifetime_correct"
] / selected_profile["lifetime_questions"].replace(0, np.nan)


selected_profile["lifetime_first_try_pct"] = selected_profile[
    "lifetime_first_answers"
] / selected_profile["lifetime_questions"].replace(0, np.nan)


scope_counts = pd.DataFrame(
    [
        {
            "scope": scope_name,
            "selected_player_questions": len(
                frame[frame["player_id"] == selected_player_id]
            ),
            "field_rows": len(frame[frame["player_id"] != selected_player_id]),
            "top_1_pct_players": max(
                build_benchmark_cohorts(frame, selected_player_id)["top_1_pct"]["size"],
                0,
            ),
            "top_10_pct_players": max(
                build_benchmark_cohorts(frame, selected_player_id)["top_10_pct"][
                    "size"
                ],
                0,
            ),
        }
        for scope_name, frame in [
            ("current_season", scope_frames["current_season"]),
            ("last_season", scope_frames["last_season"]),
            ("since_start_date", scope_frames["since_start_date"]),
        ]
    ]
)


display(
    selected_profile[
        [
            "subject",
            "player_id",
            "lifetime_questions",
            "lifetime_correct",
            "lifetime_first_answers",
            "lifetime_correct_pct",
            "lifetime_first_try_pct",
            "answer_streak",
            "season_score",
            "score",
        ]
    ]
)


display(scope_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))


player_daily = selected_rows.sort_values("sent_at").copy()

player_daily["cumulative_correct_rate"] = player_daily["solved"].cumsum() / np.arange(
    1, len(player_daily) + 1
)

player_daily["rolling_correct_rate"] = (
    player_daily["solved"].rolling(ROLLING_WINDOW_DAYS, min_periods=3).mean()
)


field_daily = peer_rows.groupby("sent_at", as_index=False).agg(
    daily_correct_rate=("solved", "mean"),
    questions_answered=("solved", "size"),
)

field_daily = field_daily.sort_values("sent_at")

field_daily["cumulative_correct_rate"] = (
    field_daily["daily_correct_rate"] * field_daily["questions_answered"]
).cumsum() / field_daily["questions_answered"].cumsum()

field_daily["rolling_correct_rate"] = (
    field_daily["daily_correct_rate"].rolling(ROLLING_WINDOW_DAYS, min_periods=3).mean()
)


axes[0].plot(
    player_daily["sent_at"],
    player_daily["cumulative_correct_rate"],
    label=TARGET_PLAYER_LABEL,
    linewidth=2.5,
)

axes[0].plot(
    field_daily["sent_at"],
    field_daily["cumulative_correct_rate"],
    label="Rest of field",
    linewidth=2.0,
)

axes[0].set_title("Cumulative correct rate over time")

axes[0].set_ylabel("Correct rate")

axes[0].legend()


axes[1].plot(
    player_daily["sent_at"],
    player_daily["rolling_correct_rate"],
    label=TARGET_PLAYER_LABEL,
    linewidth=2.5,
)

axes[1].plot(
    field_daily["sent_at"],
    field_daily["rolling_correct_rate"],
    label="Rest of field",
    linewidth=2.0,
)

axes[1].set_title(f"Rolling {ROLLING_WINDOW_DAYS}-day correct rate")

axes[1].set_ylabel("Correct rate")

axes[1].legend()


for axis in axes:

    axis.set_xlabel("Date")

    axis.set_ylim(0, 1)


plt.tight_layout()

plt.show()

In [ ]:
comparison_rows = pd.DataFrame(
    [
        {
            "slice": "All questions",
            "player_correct_rate": selected_rows["solved"].mean(),
            "field_correct_rate": peer_rows["solved"].mean(),
            "player_first_try_rate": selected_rows["first_try_correct"].mean(),
            "field_first_try_rate": peer_rows["first_try_correct"].mean(),
        },
        {
            "slice": "Challenging questions",
            "player_correct_rate": selected_rows.loc[
                selected_rows["is_challenging"], "solved"
            ].mean(),
            "field_correct_rate": peer_rows.loc[
                peer_rows["is_challenging"], "solved"
            ].mean(),
            "player_first_try_rate": selected_rows.loc[
                selected_rows["is_challenging"], "first_try_correct"
            ].mean(),
            "field_first_try_rate": peer_rows.loc[
                peer_rows["is_challenging"], "first_try_correct"
            ].mean(),
        },
        {
            "slice": "Questions with hint timestamps",
            "player_correct_rate": selected_rows.loc[
                selected_rows["hint_sent_at"].notna(), "correct_before_hint"
            ].mean(),
            "field_correct_rate": peer_rows.loc[
                peer_rows["hint_sent_at"].notna(), "correct_before_hint"
            ].mean(),
            "player_first_try_rate": selected_rows.loc[
                selected_rows["hint_sent_at"].notna(), "first_try_correct"
            ].mean(),
            "field_first_try_rate": peer_rows.loc[
                peer_rows["hint_sent_at"].notna(), "first_try_correct"
            ].mean(),
        },
    ]
)


plot_df = comparison_rows.melt(
    id_vars="slice",
    value_vars=[
        "player_correct_rate",
        "field_correct_rate",
        "player_first_try_rate",
        "field_first_try_rate",
    ],
    var_name="metric_variant",
    value_name="rate",
)


plot_df[["group", "metric"]] = plot_df["metric_variant"].str.extract(
    r"^(player|field)_(.*)$"
)

plot_df["group"] = plot_df["group"].map(
    {"player": TARGET_PLAYER_LABEL, "field": "Rest of field"}
)

plot_df["metric"] = plot_df["metric"].map(
    {
        "correct_rate": "Correct rate",
        "first_try_rate": "First-try correct rate",
    }
)


fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for axis, metric_name in zip(axes, ["Correct rate", "First-try correct rate"]):

    metric_slice = plot_df[plot_df["metric"] == metric_name]

    pivot = metric_slice.pivot(index="slice", columns="group", values="rate")

    pivot.plot(kind="bar", ax=axis, rot=15)

    axis.set_title(metric_name)

    axis.set_ylabel("Rate")

    axis.set_ylim(0, 1)


plt.tight_layout()

plt.show()

In [ ]:
difficulty_bucket_summary = pd.concat(
    [
        selected_rows.groupby("difficulty_bucket", observed=False)
        .agg(
            correct_rate=("solved", "mean"),
            first_try_rate=("first_try_correct", "mean"),
            questions=("daily_question_id", "count"),
            median_field_correct_rate=("field_correct_rate", "median"),
        )
        .assign(group=TARGET_PLAYER_LABEL),
        peer_rows.groupby("difficulty_bucket", observed=False)
        .agg(
            correct_rate=("solved", "mean"),
            first_try_rate=("first_try_correct", "mean"),
            questions=("daily_question_id", "count"),
            median_field_correct_rate=("field_correct_rate", "median"),
        )
        .assign(group="Rest of field"),
    ]
).reset_index()


display(difficulty_bucket_summary.sort_values(["difficulty_bucket", "group"]))


fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)


for axis, metric, title in [
    (axes[0], "correct_rate", "Correct rate by question difficulty"),
    (axes[1], "first_try_rate", "First-try rate by question difficulty"),
]:

    for group_name, frame in difficulty_bucket_summary.groupby("group"):

        frame = frame.sort_values("difficulty_bucket")

        axis.plot(
            frame["difficulty_bucket"].astype(str),
            frame[metric],
            marker="o",
            linewidth=2,
            label=group_name,
        )

    axis.set_title(title)

    axis.set_xlabel("Difficulty bucket (Q1 = hardest, lower field solve rate)")

    axis.set_ylabel("Rate")

    axis.set_ylim(0, 1)

    axis.legend()


plt.tight_layout()

plt.show()


player_vs_field_difficulty = selected_rows[
    [
        "sent_at",
        "daily_question_id",
        "category",
        "source",
        "value",
        "solved",
        "first_try_correct",
        "field_correct_rate",
        "difficulty_bucket",
    ]
].copy()


player_vs_field_difficulty["field_difficulty_gap_pp"] = (
    1 - player_vs_field_difficulty["field_correct_rate"]
) * 100


player_vs_field_difficulty = player_vs_field_difficulty.sort_values(
    ["field_correct_rate", "sent_at"]
)


display(
    player_vs_field_difficulty[player_vs_field_difficulty["solved"] == 1][
        [
            "sent_at",
            "daily_question_id",
            "category",
            "source",
            "value",
            "difficulty_bucket",
            "field_correct_rate",
            "field_difficulty_gap_pp",
            "first_try_correct",
        ]
    ].head(20)
)

In [ ]:
speed_summary = pd.DataFrame(
    {
        "group": [TARGET_PLAYER_LABEL, "Rest of field"],
        "median_minutes_to_first_guess": [
            selected_rows["minutes_to_first_guess"].median(),
            peer_rows["minutes_to_first_guess"].median(),
        ],
        "median_minutes_to_correct": [
            selected_rows.loc[
                selected_rows["solved"] == 1, "minutes_to_correct"
            ].median(),
            peer_rows.loc[peer_rows["solved"] == 1, "minutes_to_correct"].median(),
        ],
    }
)

display(speed_summary)


fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(
    selected_rows["minutes_to_first_guess"].dropna(),
    bins=20,
    alpha=0.65,
    label=TARGET_PLAYER_LABEL,
)

axes[0].hist(
    peer_rows["minutes_to_first_guess"].dropna(),
    bins=40,
    alpha=0.45,
    label="Rest of field",
)

axes[0].set_title("Time to first guess")

axes[0].set_xlabel("Minutes from morning message")

axes[0].legend()


axes[1].hist(
    selected_rows.loc[selected_rows["solved"] == 1, "minutes_to_correct"].dropna(),
    bins=20,
    alpha=0.65,
    label=TARGET_PLAYER_LABEL,
)

axes[1].hist(
    peer_rows.loc[peer_rows["solved"] == 1, "minutes_to_correct"].dropna(),
    bins=40,
    alpha=0.45,
    label="Rest of field",
)

axes[1].set_title("Time to correct answer")

axes[1].set_xlabel("Minutes from morning message")

axes[1].legend()


plt.tight_layout()

plt.show()

In [ ]:
def grouped_rate_table(
    df: pd.DataFrame, group_col: str, metric_col: str
) -> pd.DataFrame:
    grouped = df.groupby(group_col, dropna=False).agg(
        opportunities=("daily_question_id", "count"),
        successes=(metric_col, "sum"),
    )
    grouped["rate"] = grouped["successes"] / grouped["opportunities"]
    return grouped.sort_values("opportunities", ascending=False).reset_index()


player_source = grouped_rate_table(selected_rows, "source", "solved").rename(
    columns={"rate": "player_rate", "opportunities": "player_n"}
)
field_source = grouped_rate_table(peer_rows, "source", "solved").rename(
    columns={"rate": "field_rate", "opportunities": "field_n"}
)
source_comparison = player_source.merge(field_source, on="source", how="outer")[
    ["source", "player_n", "player_rate", "field_n", "field_rate"]
]
source_comparison["delta_pp"] = (
    source_comparison["player_rate"] - source_comparison["field_rate"]
) * 100
source_comparison = source_comparison.sort_values(
    ["player_n", "field_n"], ascending=False
)
display(source_comparison.head(15))

player_category = grouped_rate_table(selected_rows, "category", "solved").rename(
    columns={"rate": "player_rate", "opportunities": "player_n"}
)
field_category = grouped_rate_table(peer_rows, "category", "solved").rename(
    columns={"rate": "field_rate", "opportunities": "field_n"}
)
category_comparison = player_category.merge(field_category, on="category", how="outer")[
    ["category", "player_n", "player_rate", "field_n", "field_rate"]
]
category_comparison["delta_pp"] = (
    category_comparison["player_rate"] - category_comparison["field_rate"]
) * 100
category_comparison = category_comparison.sort_values(
    ["player_n", "field_n"], ascending=False
)
display(category_comparison.head(15))

## Reading the output

Useful patterns to investigate further:
- very high first-try accuracy combined with very fast solve times
- large positive gaps on challenging questions relative to the field
- abrupt jumps in rolling correctness that do not match the player's prior baseline
- unusually strong performance on specific sources or categories only

A clean next step after this notebook is to inspect raw guesses for suspicious dates, especially on days where the player beat the field by a wide margin on difficult questions.